# 01 - Data exploration

Goal: understand the Kaggle Credit Card Fraud dataset before modelling, then create a
stratified, leakage-free train/test split that every later phase reuses.

Questions this notebook answers:
1. How imbalanced are the classes, and what does that mean for evaluation?
2. What do `Time`, `Amount` and `V1`-`V28` look like, and which features separate fraud?
3. Are there duplicates that could leak between train and test?

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from ledgerwatch.data import (
    PCA_FEATURES,
    TARGET,
    drop_duplicates,
    load_raw,
    save_splits,
    split_data,
)

pd.set_option("display.max_columns", 40)
df = load_raw()
df.shape

## 1. Basic checks: types, missing values

In [ ]:
print(df.dtypes.value_counts())
print("Missing values:", int(df.isna().sum().sum()))
df.head()

## 2. Class balance

In [ ]:
counts = df[TARGET].value_counts().rename({0: "legit", 1: "fraud"})
print(counts)
print(f"Fraud rate: {df[TARGET].mean():.4%}")
print(f"Accuracy of always predicting 'legit': {1 - df[TARGET].mean():.4%}")

A model that never flags anything gets ~99.8% accuracy and catches zero fraud, so accuracy
is useless here. We will use:

- **Precision**: of the transactions we flag, how many are really fraud (analyst workload).
- **Recall**: of all fraud, how much we catch (money lost).
- **F1**: balance of the two.
- **PR-AUC**: precision/recall across all thresholds. Unlike ROC-AUC, it is not inflated
  by the huge number of easy true negatives.

## 3. `Amount`

In [ ]:
print(df.groupby(TARGET)["Amount"].describe())

fig, ax = plt.subplots(figsize=(8, 4))
for label, name in [(0, "legit"), (1, "fraud")]:
    ax.hist(df.loc[df[TARGET] == label, "Amount"].clip(lower=0.01),
            bins=50, alpha=0.6, density=True, label=name, log=False)
ax.set_xscale("log")
ax.set_xlabel("Amount (log scale)")
ax.set_ylabel("density")
ax.legend()
ax.set_title("Transaction amount by class")
plt.show()

## 4. `Time` and hour of day

`Time` is seconds since the first transaction in the dataset (about 2 days). It is not a
real clock time, but `(Time // 3600) % 24` gives an approximate hour, which is useful for
the "odd hours" rule in Phase 5.

In [ ]:
hours = (df["Time"] // 3600 % 24).astype(int)
fraud_rate_by_hour = df.groupby(hours)[TARGET].mean()
volume_by_hour = hours.value_counts().sort_index()

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.bar(volume_by_hour.index, volume_by_hour.values, color="lightgray", label="volume")
ax1.set_xlabel("hour (approx.)")
ax1.set_ylabel("transactions")
ax2 = ax1.twinx()
ax2.plot(fraud_rate_by_hour.index, fraud_rate_by_hour.values, color="crimson", marker="o")
ax2.set_ylabel("fraud rate", color="crimson")
ax1.set_title("Volume and fraud rate by hour")
plt.show()

## 5. `V1`-`V28`

These are PCA components. The bank anonymized the original features, so we cannot say what
`V14` *means*, only how it behaves. Below: which components differ most between classes.

In [ ]:
corr = df[PCA_FEATURES + [TARGET]].corr()[TARGET].drop(TARGET)
top = corr.abs().sort_values(ascending=False).head(8)
print(corr[top.index].round(3))

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, feat in zip(axes, top.index[:4], strict=True):
    df.boxplot(column=feat, by=TARGET, ax=ax, showfliers=False)
    ax.set_title(feat)
    ax.set_xlabel("Class")
plt.suptitle("Top 4 PCA features by |correlation| with fraud")
plt.tight_layout()
plt.show()

## 6. Duplicates

In [ ]:
dup_mask = df.duplicated()
print(f"Exact duplicate rows: {dup_mask.sum():,}")
print(f"Duplicate fraud rows: {int(df.loc[dup_mask, TARGET].sum())}")

If a duplicate lands in both train and test, the model is graded on a row it memorized.
We drop exact duplicates **before** splitting.

## 7. Stratified train/test split

- **Stratified** so both sides keep the same fraud rate (with ~470 fraud rows, a random split
  could leave the test set noticeably short of fraud).
- **No scaling here.** Any scaler is fit later inside a scikit-learn `Pipeline` on the training
  set only. Fitting it on all data would leak test statistics into training.
- The split is saved once to `data/processed/` with a SHA-256 of the raw file, so every model is
  evaluated on exactly the same test set.

In [ ]:
deduped = drop_duplicates(df)
train, test = split_data(deduped)

summary = pd.DataFrame({
    "rows": [len(train), len(test)],
    "fraud": [int(train[TARGET].sum()), int(test[TARGET].sum())],
    "fraud_rate": [train[TARGET].mean(), test[TARGET].mean()],
}, index=["train", "test"])
print(summary)

overlap = train.merge(test, how="inner")
print("Rows appearing in both train and test:", len(overlap))

In [ ]:
meta = save_splits(train, test)
meta

## 8. Findings (write these yourself)

Fill this in after running the notebook. One or two sentences each, using the real numbers:

- Class balance:
- Amount:
- Hour of day:
- Most informative PCA features:
- Duplicates:
- Split: